In [ ]:
# Cell 1: Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Cell 2: Load Titanic dataset once
df = sns.load_dataset("titanic")
df.to_csv("titanic.csv", index=False)  # offline fallback
print(df.shape)
df.info()
df.describe()

# Cell 3: Missing values percentage
missing = df.isnull().mean() * 100
print(missing[missing > 0])

# Cell 4: Handle missing values (threshold rule)
# Example: drop rows if <5%, impute if 5–30%, drop/encode if >30%
df_clean = df.copy()
df_clean["age"].fillna(df_clean["age"].median(), inplace=True)
df_clean.dropna(subset=["embarked"], inplace=True)

# Cell 5: Univariate analysis
plt.hist(df_clean["age"], bins=20); plt.title("Age Distribution"); plt.show()
plt.boxplot(df_clean["age"].dropna()); plt.title("Age Boxplot"); plt.show()

# Outlier detection using IQR
Q1, Q3 = df_clean["fare"].quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = ((df_clean["fare"] < Q1 - 1.5*IQR) | (df_clean["fare"] > Q3 + 1.5*IQR)).sum()
print("Fare outliers:", outliers)

# Cell 6: Bivariate survival analysis
print("Survival by sex:\n", df_clean.groupby("sex")["survived"].mean())
print("Survival by pclass:\n", df_clean.groupby("pclass")["survived"].mean())
print("Survival by sex+pclass:\n", df_clean.groupby(["sex","pclass"])["survived"].mean())

# Cell 7: Correlation matrix
cols = ["survived","pclass","age","sibsp","parch","fare"]
corr = df_clean[cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.show()

# Cell 8: Multivariate charts (examples)
sns.barplot(x="sex", y="survived", data=df_clean)
plt.title("Survival by Sex"); plt.show()

sns.barplot(x="pclass", y="survived", data=df_clean)
plt.title("Survival by Class"); plt.show()

sns.scatterplot(x="age", y="fare", hue="survived", data=df_clean)
plt.title("Age vs Fare by Survival"); plt.show()

sns.boxplot(x="sex", y="age", hue="survived", data=df_clean)
plt.title("Age distribution by Sex and Survival"); plt.show()

# Cell 9: Standardization check
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_clean[["age_z","fare_z"]] = scaler.fit_transform(df_clean[["age","fare"]])
print(df_clean[["age","age_z","fare","fare_z"]].describe())
